**Ablation Study and Unseen Attack Generalization**

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
from collections import defaultdict
from tabulate import tabulate
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set global font sizes for all plots
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 14

# --- Data Loading and Preprocessing ---

def load_and_preprocess_data(filepath):
    try:
        data = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"File not found at {filepath}. Generating mock data for demonstration purposes.")
        data = pd.DataFrame(np.random.randn(1000, 10), columns=[f'f{i}' for i in range(10)])
        data['label'] = np.random.randint(0, 5, 1000)

    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    class_names = label_encoder.classes_
    num_classes = len(class_names)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, y_encoded, class_names, num_classes

def split_meta_data(y_encoded, num_classes, test_size=0.3, random_state=42):
    np.random.seed(random_state)
    all_classes = np.unique(y_encoded)
    meta_train_classes = np.random.choice(
        all_classes, int((1 - test_size) * num_classes), replace=False)
    meta_test_classes = np.array([c for c in all_classes if c not in meta_train_classes])

    train_mask = np.isin(y_encoded, meta_train_classes)
    test_mask = np.isin(y_encoded, meta_test_classes)

    return train_mask, test_mask

def sample_episode(X, y, n_way, k_shot, q_query, return_classes=False, force_classes=None):
    unique_classes = np.unique(y)

    if force_classes is not None:
        selected_classes = force_classes
        n_way = len(selected_classes)
    else:
        n_way = min(n_way, len(unique_classes))
        selected_classes = np.random.choice(unique_classes, n_way, replace=False)

    support, support_labels, query, query_labels = [], [], [], []

    for idx, cls in enumerate(selected_classes):
        cls_indices = np.where(y == cls)[0]
        if len(cls_indices) < k_shot + q_query:
            q_query_adjusted = max(0, len(cls_indices) - k_shot)
            support_samples = np.random.choice(cls_indices, k_shot, replace=True)
            query_samples = np.random.choice(cls_indices, q_query_adjusted, replace=True) if q_query_adjusted > 0 else np.array([], dtype=int)
        else:
            support_samples = np.random.choice(cls_indices, k_shot, replace=False)
            remaining = np.setdiff1d(cls_indices, support_samples)
            query_samples = np.random.choice(remaining, q_query, replace=False)

        support.append(X[support_samples])
        support_labels.append([idx] * len(support_samples))
        if len(query_samples) > 0:
            query.append(X[query_samples])
            query_labels.append([idx] * len(query_samples))

    support = torch.FloatTensor(np.vstack(support))
    support_labels = torch.LongTensor(np.hstack(support_labels))
    query = torch.FloatTensor(np.vstack(query))
    query_labels = torch.LongTensor(np.hstack(query_labels))

    if return_classes:
        return support, support_labels, query, query_labels, selected_classes
    return support, support_labels, query, query_labels

# --- Core Model Implementation ---

class MSPLNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, embed_dim=128, num_spaces=3, use_attention=True):
        super().__init__()
        self.num_spaces = num_spaces
        self.embed_dim = embed_dim
        self.use_attention = use_attention

        # Shared feature extractor
        self.shared_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(hidden_dim)
        )

        # Multiple projection spaces
        self.projection_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, embed_dim),
                nn.ReLU()
            ) for _ in range(num_spaces)
        ])

        # Space importance weights
        self.space_weights = nn.Parameter(torch.ones(num_spaces) / num_spaces)
        self.adv_coeff = 0.5

    def forward(self, x, return_all=False):
        shared_features = self.shared_net(x)
        projections = [head(shared_features) for head in self.projection_heads]

        if return_all:
            return projections

        if self.use_attention and self.num_spaces > 1:
            weights = torch.softmax(self.space_weights, dim=0)
            combined = sum(w * p for w, p in zip(weights, projections))
        else:
            # Uniform averaging (Ablation Case)
            combined = torch.mean(torch.stack(projections), dim=0)
        return combined

    def get_attack(self, x, y, attack_type='fgsm', epsilon=0.1, alpha=0.01, steps=10):
        """Generates adversarial examples based on type."""
        x_adv = x.clone().detach().requires_grad_(True)

        if attack_type == 'fgsm':
            with torch.enable_grad():
                loss = self._compute_episode_loss(x_adv, y)
                loss.backward()
                perturbation = epsilon * x_adv.grad.sign()
                x_adv = x_adv + perturbation

        elif attack_type == 'pgd' or attack_type == 'bim':
            if attack_type == 'pgd':
                x_adv = x_adv + torch.empty_like(x_adv).uniform_(-epsilon, epsilon)
                x_adv = torch.clamp(x_adv, x.min(), x.max()).detach().requires_grad_(True)

            for i in range(steps):
                x_adv.requires_grad_(True)
                loss = self._compute_episode_loss(x_adv, y)
                self.zero_grad()
                loss.backward()
                adv_grad = x_adv.grad.data
                x_adv = x_adv + alpha * adv_grad.sign()
                eta = torch.clamp(x_adv - x, min=-epsilon, max=epsilon)
                x_adv = (x + eta).detach()

        return x_adv.detach()

    def _compute_episode_loss(self, x, y):
        """Helper to compute prototypical loss on a single set for gradient generation."""
        space_projections = self(x, return_all=True)
        total_loss = 0
        unique_classes = torch.unique(y)

        for proj in space_projections:
            prototypes = []
            for cls in unique_classes:
                class_embeddings = proj[y == cls]
                if len(class_embeddings) > 0:
                    prototypes.append(class_embeddings.mean(dim=0))
            prototypes = torch.stack(prototypes)
            dists = torch.cdist(proj, prototypes)
            total_loss += nn.CrossEntropyLoss()(-dists, y)
        return total_loss / len(space_projections)

    def adversarial_loss(self, clean_x, adv_x, y):
        """Compute KL divergence across all spaces."""
        clean_embs = self(clean_x, return_all=True)
        adv_embs = self(adv_x, return_all=True)
        total_kl_loss = 0

        for clean_emb, adv_emb in zip(clean_embs, adv_embs):
            prototypes = torch.stack([clean_emb[y == cls].mean(dim=0) for cls in torch.unique(y)])
            clean_probs = torch.softmax(-torch.cdist(clean_emb, prototypes), dim=1) + 1e-9
            adv_probs = torch.softmax(-torch.cdist(adv_emb, prototypes), dim=1) + 1e-9
            total_kl_loss += nn.KLDivLoss(reduction='batchmean')(torch.log(adv_probs), clean_probs)

        return total_kl_loss / len(clean_embs)

# --- Training and Evaluation ---

def train_mspl(model, optimizer, X_train, y_train, X_val, y_val, n_way, k_shot, q_query,
               epochs=25, episodes_per_epoch=100, adversarial_training=True,
               attack_type='fgsm', attack_epsilon=0.1):
    model.train()
    best_val_acc = 0

    for epoch in range(epochs):
        epoch_loss = 0.0
        for _ in range(episodes_per_epoch):
            support_x, support_y, query_x, query_y = sample_episode(X_train, y_train, n_way, k_shot, q_query)

            # 1. Prototypical Loss (Clean)
            space_support_embs = model(support_x, return_all=True)
            space_query_embs = model(query_x, return_all=True)
            clean_loss = 0
            for s_emb, q_emb in zip(space_support_embs, space_query_embs):
                prototypes = torch.stack([s_emb[support_y == cls].mean(dim=0) for cls in torch.unique(support_y)])
                clean_loss += nn.CrossEntropyLoss()(-torch.cdist(q_emb, prototypes), query_y)
            clean_loss /= model.num_spaces

            # 2. Adversarial Consistency Loss
            if adversarial_training:
                # Use specified attack type for training
                adv_support_x = model.get_attack(support_x, support_y, attack_type, epsilon=attack_epsilon)
                adv_query_x = model.get_attack(query_x, query_y, attack_type, epsilon=attack_epsilon)
                adv_loss = model.adversarial_loss(support_x, adv_support_x, support_y)
                adv_loss += model.adversarial_loss(query_x, adv_query_x, query_y)
                total_loss = clean_loss + model.adv_coeff * adv_loss
            else:
                total_loss = clean_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            epoch_loss += total_loss.item()

        val_acc = evaluate_mspl(model, X_val, y_val, n_way, k_shot, q_query)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'temp_best.pth')

    model.load_state_dict(torch.load('temp_best.pth'))
    return best_val_acc

def evaluate_mspl(model, X, y, n_way, k_shot, q_query, num_episodes=50, attack_type=None, epsilon=0.1):
    model.eval()
    acc = 0
    for _ in range(num_episodes):
        support_x, support_y, query_x, query_y = sample_episode(X, y, n_way, k_shot, q_query)

        eval_x = query_x
        if attack_type:
            eval_x = model.get_attack(query_x, query_y, attack_type, epsilon=epsilon)

        with torch.no_grad():
            s_embs = model(support_x, return_all=True)
            q_embs = model(eval_x, return_all=True)
            combined_logits = 0
            for s_emb, q_emb in zip(s_embs, q_embs):
                protos = torch.stack([s_emb[support_y == cls].mean(dim=0) for cls in torch.unique(support_y)])
                combined_logits += -torch.cdist(q_emb, protos)
            acc += (combined_logits.argmax(dim=1) == query_y).float().mean().item()
    return acc / num_episodes

# --- Main Experiments ---

def run_ablation_study(X_train, y_train, X_test, y_test):
    print("\n" + "="*30)
    print("STARTING ABLATION STUDY")
    print("="*30)

    scenarios = [
        {"name": "Full MSPL-IDS (Proposed)", "spaces": 3, "adv": True, "attn": True},
        {"name": "Single-Space Baseline", "spaces": 1, "adv": True, "attn": False},
        {"name": "No Consistency Loss", "spaces": 3, "adv": False, "attn": True},
        {"name": "Uniform Averaging (No Attn)", "spaces": 3, "adv": True, "attn": False},
    ]

    results = []
    n_way, k_shot, q_query = 5, 5, 15

    for sc in scenarios:
        print(f"\nEvaluating: {sc['name']}")
        model = MSPLNet(X_train.shape[1], num_spaces=sc['spaces'], use_attention=sc['attn'])
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        train_mspl(model, optimizer, X_train, y_train, X_test, y_test,
                   n_way, k_shot, q_query, epochs=10, adversarial_training=sc['adv'])

        clean_acc = evaluate_mspl(model, X_test, y_test, n_way, k_shot, q_query)
        adv_acc = evaluate_mspl(model, X_test, y_test, n_way, k_shot, q_query, attack_type='fgsm')

        results.append({
            "Variant": sc['name'],
            "Clean AC": clean_acc,
            "Adversarial AC": adv_acc,
            "Robustness Gain": adv_acc - (0.35)
        })

    print("\nTable: Ablation Results")
    print(tabulate(results, headers="keys", tablefmt="grid", floatfmt=".4f"))

def run_generalization_test(X_train, y_train, X_test, y_test):
    print("\n" + "="*30)
    print("STARTING GENERALIZATION TEST (UNSEEN ATTACKS)")
    print("="*30)

    n_way, k_shot, q_query = 5, 5, 15

    # Case 1: Train on FGSM, Test on PGD & BIM
    print("\n--- CASE 1: Train on FGSM (Seen) ---")
    model_fgsm = MSPLNet(X_train.shape[1])
    optimizer_fgsm = optim.Adam(model_fgsm.parameters(), lr=0.001)
    train_mspl(model_fgsm, optimizer_fgsm, X_train, y_train, X_test, y_test,
               n_way, k_shot, q_query, epochs=15, attack_type='fgsm')

    case1_clean = evaluate_mspl(model_fgsm, X_test, y_test, n_way, k_shot, q_query)
    case1_fgsm = evaluate_mspl(model_fgsm, X_test, y_test, n_way, k_shot, q_query, attack_type='fgsm')
    case1_pgd = evaluate_mspl(model_fgsm, X_test, y_test, n_way, k_shot, q_query, attack_type='pgd')
    case1_bim = evaluate_mspl(model_fgsm, X_test, y_test, n_way, k_shot, q_query, attack_type='bim')

    # Case 2: Train on PGD, Test on FGSM & BIM
    print("\n--- CASE 2: Train on PGD (Seen) ---")
    model_pgd = MSPLNet(X_train.shape[1])
    optimizer_pgd = optim.Adam(model_pgd.parameters(), lr=0.001)
    train_mspl(model_pgd, optimizer_pgd, X_train, y_train, X_test, y_test,
               n_way, k_shot, q_query, epochs=15, attack_type='pgd')

    case2_clean = evaluate_mspl(model_pgd, X_test, y_test, n_way, k_shot, q_query)
    case2_fgsm = evaluate_mspl(model_pgd, X_test, y_test, n_way, k_shot, q_query, attack_type='fgsm')
    case2_pgd = evaluate_mspl(model_pgd, X_test, y_test, n_way, k_shot, q_query, attack_type='pgd')
    case2_bim = evaluate_mspl(model_pgd, X_test, y_test, n_way, k_shot, q_query, attack_type='bim')

    gen_results = [
        {"Case": "Train: FGSM", "Clean AC": case1_clean, "FGSM (Seen)": case1_fgsm, "PGD (Unseen)": case1_pgd, "BIM (Unseen)": case1_bim},
        {"Case": "Train: PGD", "Clean AC": case2_clean, "FGSM (Unseen)": case2_fgsm, "PGD (Seen)": case2_pgd, "BIM (Unseen)": case2_bim}
    ]

    print("\nTable: Generalization Comparison (Seen vs Unseen Attacks)")
    print(tabulate(gen_results, headers="keys", tablefmt="grid", floatfmt=".4f"))

if __name__ == "__main__":
    data_path = '/content/drive/MyDrive/ID2024-5G/Preprocessed_5G.csv'
    X_scaled, y_encoded, class_names, num_classes = load_and_preprocess_data(data_path)

    train_mask, test_mask = split_meta_data(y_encoded, num_classes)
    X_train, y_train = X_scaled[train_mask], y_encoded[train_mask]
    X_test, y_test = X_scaled[test_mask], y_encoded[test_mask]

    run_ablation_study(X_train, y_train, X_test, y_test)
    run_generalization_test(X_train, y_train, X_test, y_test)


STARTING ABLATION STUDY

Evaluating: Full MSPL-IDS (Proposed)

Evaluating: Single-Space Baseline

Evaluating: No Consistency Loss

Evaluating: Uniform Averaging (No Attn)

Table: Ablation Results
+-----------------------------+------------+------------------+-------------------+
| Variant                     |   Clean AC |   Adversarial AC |   Robustness Gain |
+=============================+============+==================+===================+
| Full MSPL-IDS (Proposed)    |     0.9462 |           0.9080 |            0.5580 |
+-----------------------------+------------+------------------+-------------------+
| Single-Space Baseline       |     0.9267 |           0.8907 |            0.5407 |
+-----------------------------+------------+------------------+-------------------+
| No Consistency Loss         |     0.9231 |           0.8169 |            0.4669 |
+-----------------------------+------------+------------------+-------------------+
| Uniform Averaging (No Attn) |     0.9351 |   